In [2]:
%load_ext dotenv
%dotenv

# import api_key from .env
import os
import random
from io import StringIO
import pandas as pd
from google import genai
from google.genai import types

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [3]:

client = genai.Client(api_key='') 

In [93]:
# as gpt-4 does not work with system prompt, use this
instruction_Baseline_for_gpt4 = """You are a professional annotator tasked with categorizing social media posts.
Assign a binary code to each post - whether (code 1) or not (code 0) the post reflects the Values listed below:
Self-direction: Independent thought and action—choosing, creating, and exploring.
Stimulation: Excitement, novelty, and challenge in life.
Hedonism: Pleasure and sensuous gratification for oneself.
Achievement: Personal success through demonstrating competence according to social standards.
Power: Social status and prestige, wealth, control or dominance over people and resources.
Security: Safety, harmony, strong government, and stability of society, relationships, and self.
Conformity: The restraint of actions, inclinations, and impulses that are likely to upset or harm others and violate social expectations or norms.
Tradition: Respect, commitment, and acceptance of the customs and ideas that traditional culture or religion provides.
Benevolence: Preservation and enhancement of the welfare of people with whom one is in frequent personal contact.
Universalism: Understanding, appreciation, tolerance, and protection for the welfare of all people and of nature.
For example, the post "Для меня главное традиции и друзья" expresses Value Tradition and Value Benevolence.
For the following JSON entries of type {post_id:post_text}, return comma separated data frame with post_id as index, Value as columns and binary code in cell.
"""

# for gpt-4 use this:
# instruction_old = """For the following JSON entries of type {post_id:post_text}, return comma separated data frame with post_id as index, Value as columns and binary code in cell.
# Posts: 
# """
# instruction_end_old="""
# Print comma separated data frame only without any explanation. Before printing, ensure that each post_id and each Value in the data frame has a corresponding binary code value. Data frame must have all post_id as index and all Values as columns. If errors are found, fix them."""

In [1]:
instruction = """For the following JSON entries of type {post_id:post_text}, return annotations for each post_id in a single JSON object.
Posts:
"""
# instruction_RU = """Для следующих JSON-записей вида {post_id: post_text} верни аннотации для каждого post_id в виде одного JSON-объекта.
# Посты:
# """


In [128]:
system_prompt_Baseline = """You are a professional annotator tasked with categorizing social media posts.
Assign a binary code to each post - whether (code 1) or not (code 0) the post reflects the Values listed below:
Self-direction: Independent thought and action—choosing, creating, and exploring.
Stimulation: Excitement, novelty, and challenge in life.
Hedonism: Pleasure and sensuous gratification for oneself.
Achievement: Personal success through demonstrating competence according to social standards.
Power: Social status and prestige, wealth, control or dominance over people and resources.
Security: Safety, harmony, strong government, and stability of society, relationships, and self.
Conformity: The restraint of actions, inclinations, and impulses that are likely to upset or harm others and violate social expectations or norms.
Tradition: Respect, commitment, and acceptance of the customs and ideas that traditional culture or religion provides.
Benevolence: Preservation and enhancement of the welfare of people with whom one is in frequent personal contact.
Universalism: Understanding, appreciation, tolerance, and protection for the welfare of all people and of nature.
For example, the post "Для меня главное традиции и друзья" expresses Value Tradition and Value Benevolence.
"""

In [3]:
# to add context add this:
# You are a professional annotator of Russian social media VKontakte born in Russia and living in Russia.

In [250]:
system_Extended = """You are a professional annotator of social media posts.
Assign a binary code to each post - whether (code 1) or not (code 0) the post reflects basic human values.
Values are an expression of the importance of an object, phenomenon, or quality. The Values are listed below:
Self-direction: Importance of independent thought and action, making autonomous decisions based on one’s own judgment, creating and exploring, being curious.
Stimulation: Excitement, novelty, and variety in life, daring. Appreciation of surprises, adventures, trying new things.
Hedonism: Importance of having good time, pleasure and sensuous gratification, 'spoiling' themselves, food, leisure, sex or other personally enjoyable activities.
Achievement: Personal success through demonstrating competence according to social standards, importance of being ambitious, capable, influential, getting social recognition.
Power: Social status and prestige, wealth, control or dominance over people and resources, authority, social power. Importance of being visibly rich and in charge.
Security: Safety and harmony understood as protection from threats, strong government, stability of relationships, of self, of society and social order, national security, clean and healthy as protection from risks.
Conformity: Avoiding actions, and impulses that violate social expectations, law, or norm; to be obedient, responsible, and polite.
Tradition: Respect, commitment, and acceptance of the customs and ideas that traditional culture or religion provides, humble, devout, moderate, subordination.
Benevolence: Preservation and enhancement of the welfare of close people with whom one is in frequent personal contact (e.g., family, friends), helpful, honest, forgiving, responsible toward close others, loyal, true friendship, mature love.
Universalism: Understanding and tolerance toward people beyond one’s immediate circle, and protection for the welfare of all people in the world including strangers; importance of broadmindedness, social justice, equality, world at peace. Importance of nature and environment protection.
When assigning values, prioritize conceptual meaning over surface-level lexical cues.
Don't overestimate emotional texts.
If the value is expressed subtly or implicitly, assign it when there is reasonable textual evidence, rather than requiring explicit keywords.
Do NOT make assumptions about the author and intentions beyond the text.
Power, achievement, hedonism, stimulation, and self-direction primarily focus on personal interests and characteristics.
Benevolence, universalism, tradition, conformity, and security are primarily concerned with how one relates socially to others and affects their interests.
Power, achievement, tradition, conformity, and security serve to cope with anxiety due to uncertainty in the social and physical world.
Hedonism, stimulation, self-direction, universalism, and benevolence express anxiety-free motivations.
Security and conformity aim to avoid or overcome actual or potential danger.
Self-direction and close values motivate intrinsically rewarding social, intellectual, and emotional opportunities.

The output format is a strict contract.
Always produce exactly one CSV table with one row per post_id and exactly 10 binary value columns, no missing or extra columns.
If the format is incorrect, regenerate the output internally before responding.

"""

In [5]:
system_prompt_Bias_calibrated = """You are a professional annotator of social media posts.
Assign a binary code to each post - whether (code 1) or not (code 0) the post reflects basic human values.
Values are an expression of the importance of an object, phenomenon, or quality. The Values are listed below:
Self-direction: Importance of independent thought and action, making autonomous decisions based on one’s own judgment, creating and exploring, being curious.
Stimulation: Excitement, novelty, and variety in life, daring. Appreciation of surprises, adventures, trying new things.
Hedonism: Importance of having good time, pleasure and sensuous gratification, 'spoiling' themselves, food, leisure, sex or other personally enjoyable activities.
Achievement: Personal success through demonstrating competence according to social standards, importance of being ambitious, capable, influential, getting social recognition.
Power: Social status and prestige, wealth, control or dominance over people and resources, authority, social power. Importance of being visibly rich and in charge.
Security: Safety, harmony, strong government, stability of relationships, of self, of society and social order, national security, clean, healthy.
Conformity: The restraint of actions, inclinations, and impulses that are likely to upset or harm others and violate social expectations or norms.
Tradition: Respect, commitment, and acceptance of the customs and ideas that traditional culture or religion provides, humble, devout, moderate, subordination.
Benevolence: Preservation and enhancement of the welfare of close people with whom one is in frequent personal contact (e.g., family, friends), helpful, honest, forgiving, responsible toward close others, loyal, true friendship, mature love.
Universalism: Understanding, appreciation, tolerance, and protection for the welfare of all people in the world including strangers; importance of broadmindedness, social justice, equality, world at peace. Importance of nature and environment protection.
When assigning values, prioritize conceptual meaning over surface-level lexical cues.
Don't overestimate emotional texts.
If the value is expressed subtly or implicitly, assign it when there is reasonable textual evidence, rather than requiring explicit keywords.
Do NOT make assumptions about the author and intentions beyond the text.
Power, achievement, hedonism, stimulation, and self-direction primarily focus on personal interests and characteristics.
Benevolence, universalism, tradition, conformity, and security are primarily concerned with how one relates socially to others and affects their interests.
Power, achievement, tradition, conformity, and security serve to cope with anxiety due to uncertainty in the social and physical world.
Hedonism, stimulation, self-direction, universalism, and benevolence express anxiety-free motivations.
Security and conformity aim to avoid or overcome actual or potential danger.
Self-direction and close values motivate intrinsically rewarding social, intellectual, and emotional opportunities.

Return the result strictly in JSON according to the provided schema.
"""

In [5]:
system_prompt_Bias_calibrated_RU = """Ты — профессиональный аннотатор постов в социальных сетях.
Для каждого поста присвой бинарную метку: отражает ли пост важность каждой из ценностей (код 1) или нет (код 0).
Ценности  – это выражение важности какого-либо объекта, явления или качества.  Список ценностей и их определения приведены ниже:
Самостоятельность (Self-direction): важность независимого мышления и действий, принятие автономных решений, собственные суждение, личный выбор, поиск, творчество и любознательность.
Риск-Новизна (Stimulation): жизненный азарт, стремление к новизне и разнообразию. Ценность острых ощущений, приключений и экспериментирования.
Гедонизм (Hedonism): важность того, чтобы просто наслаждаться жизнью, получать удовольствие, баловать себя, вкусно есть и хорошо отдыхать.
Достижение (Achievement): личный успех, проявляющийся через демонстрацию собственной состоятельности в соответствии с общественными ожиданиями /стандартами. Важность достижения целей, получения заслуженных наград и признания окружающих.
Власть-Богатство (Power): социальный статус и престиж, богатство, контроль и доминирование над людьми и ресурсами, авторитет, социальная власть. Важность быть видимо богатым и влиятельным.
Безопасность (Security): защищенность, гармония, сильное государство, стабильность общества и отношений, последовательность, неизменность собственной личности, общества и социального порядка, национальная безопасность, чистота, здоровье.
Традиция (Tradition): уважительное отношение к окружающим, следование обычаям и традициям, предлагаемым культурой или религией, скромность, набожность, сдержанность, подчинение.
Конформность (Conformity): сдерживание поступков, склонностей и побуждений, которые могут нарушить общественные ожидания, правила, законы, или причинить кому-то вред.
Благожелательность (Benevolence): сохранение и приумножение благополучия близких людей, с которыми человек находится в частом личном общении (например, семья, друзья), готовность помочь, честность, умение прощать, ответственность по отношению к близким, верность, настоящая дружба, зрелая любовь.
Универсализм (Universalism): понимание, признание, толерантность, важность защиты благополучия и равенства прав всех людей, включая незнакомых; важность широты взглядов, социальной справедливости, равенства, мира во всем мире. Важность защиты природы и окружающей среды.

Власть-богатство, достижение, гедонизм, риск-новизна и самостоятельность в первую очередь подчеркивают личные интересы и характеристики.
Благожелательность, универсализм, традиция, конформность и безопасность поддерживают интересы группы и других людей.
Власть-богатство, достижение, традиция, конформность и безопасность связаны  с тревогой и неопределенностью.
Гедонизм, риск-новизна, самостоятельность, универсализм и благожелательность выражают мотивации, не связанные с тревогой.
Безопасность и конформность направлены на предотвращение или преодоление реальной или потенциальной опасности.
Самостоятельность и универсализм направлены на развитие и рост.
При присвоении ценностей ориентируйся в первую очередь на смысл текста, а не на отдельные слова.
Не переоценивай эмоциональные тексты.
Если ценность выражена косвенно или неявно, присваивай ее при наличии обоснованных доказательств в тексте, а не требуй явных ключевых слов.
НЕ делай предположений об авторе и его намерениях, выходящих за рамки текста
Верни результат строго в формате JSON в соответствии с предоставленной схемой.
"""

In [6]:
# instruction_end="""
# Output a CSV table only (no explanations). Header must be exactly:
# post_id,Self-direction,Stimulation,Hedonism,Achievement,Power,Security,Conformity,Tradition,Benevolence,Universalism
# Then output one row per post_id with exactly 11 comma-separated fields: post_id followed by 10 binary values in this exact order. Before responding, verify that EVERY row has 11 fields; otherwise regenerate the entire table.
# """

instruction_end = """
Output ONE JSON object only. The JSON must have exactly one key: "r". "r" is a list of rows; one row per post_id from the input.
Each row must be an array of exactly 11 integers in this exact order:
post_id, Self-direction, Stimulation, Hedonism, Achievement, Power, Security, Conformity, Tradition, Benevolence, Universalism.
"""

instruction_end_RU = """
Верни ТОЛЬКО ОДИН JSON-объект. В JSON должен быть ровно один ключ: "r".
"r" — это список строк; по одной строке на каждый post_id из входных данных.
Каждая строка должна быть массивом из ровно 11 целых чисел в следующем порядке:
post_id, Self-direction, Stimulation, Hedonism, Achievement, Power, Security, Conformity, Tradition, Benevolence, Universalism.
"""



In [11]:
# load df
df['text'] = df['text'].str.replace('"', '', regex=True)
test_=df[0:300]  #  process in steps of 300-500 posts
test__=test_.reset_index() # renumerate
test1=test__.drop('level_0', axis=1)
test1

In [13]:
import tiktoken

def num_tokens_from_messages(messages, model):  #="gpt-3.5-turbo-0613"
    """Return the number of tokens used by a list of messages."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # print("Warning: model not found. Using cl100k_base encoding.")
        encoding = tiktoken.get_encoding("cl100k_base")
    
    if model in {
        "gpt-3.5-turbo-0613",
        "gpt-3.5-turbo-16k-0613",
        "gpt-4-0314",
        "gpt-4-32k-0314",
        "gpt-4-0613",
        "gpt-4-32k-0613",
        "gpt-4-turbo",
        "gpt-4-turbo-2024-04-09",
        "gpt-4o-2024-05-13",
        }:
        tokens_per_message = 3
        tokens_per_name = 1
    elif model == "gpt-3.5-turbo-0301":
        tokens_per_message = 4  # every message follows <|start|>{role/name}\n{content}<|end|>\n
        tokens_per_name = -1  # if there's a name, the role is omitted
    elif "gpt-3.5-turbo" in model:
        print("Warning: gpt-3.5-turbo may update over time. Returning num tokens assuming gpt-3.5-turbo-0613.")
        return num_tokens_from_messages(messages, model="gpt-3.5-turbo-0613")
    elif "gpt-4o" in model:
        # print(
        #     "Warning: gpt-4o may update over time. Returning num tokens assuming gpt-4o-2024-05-13.")
        return num_tokens_from_messages(messages, model="gpt-4o-2024-05-13")
    elif "gpt-4" in model:
        print("Warning: gpt-4 may update over time. Returning num tokens assuming gpt-4-0613.")
        return num_tokens_from_messages(messages, model="gpt-4-0613")
    elif model == "gpt-5":
        return num_tokens_from_messages(messages, model="gpt-4o-2024-05-13")
    elif model.startswith("gemini"):
        encoding = tiktoken.get_encoding("cl100k_base")
        tokens_per_message = 3
        tokens_per_name = 1

    
    else:
        raise NotImplementedError(
            f"""num_tokens_from_messages() is not implemented for model {model}. See https://github.com/openai/openai-python/blob/main/chatml.md for information on how messages are converted to tokens."""
        )
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3  # every reply is primed with <|start|>assistant<|message|>
    return num_tokens

In [14]:
def SelectRandomExcept(df, idx_done, k):
    """Return list of idx, randomized. Max len = k"""
    from_which_list_to_select=set(range(0, df.shape[0]))-set(idx_done)
    k = min(k,len(from_which_list_to_select))
    random_chunk=random.sample(list(from_which_list_to_select), k)
    return [i for i in random_chunk if not i in idx_done]

In [15]:
def create_post_list_inline(instruction, instruction_end, post_dict100, model):  # returns a string with posts in random order
    instruction_num_token=num_tokens_from_messages(messages = [{"role": "user", "content": instruction}], model=model) 
    max_input_tokens=4096   #1000 - for 3.5
    post_num_token=0
    # post_list_inline=''
    dict_full={}
    # i=0  !!
    key_list=[]
    for key, post in post_dict100.items():
        #post_full='Post'+str(i+1) +': ' + '"' + post + '".'
        post_dict={}
        # post_dict[i]=post !!
        post_dict[key]=post  #new
        post_num_token=post_num_token+num_tokens_from_messages(messages = [{"role": "user", "content": str(post_dict)+instruction_end}], model=model)
        
        if instruction_num_token+post_num_token<max_input_tokens:
                dict_full.update(post_dict)
                key_list.append(key)
        if (instruction_num_token+post_num_token>max_input_tokens)&(i==0):                
                dict_full.update(post_dict)
                key_list.append(key)
        if (instruction_num_token+post_num_token>max_input_tokens)&(i>0):
            break
        #i+=1 !!
    #post_list_inline=' '.join(x for x in post_list_full)
    
    return dict_full, key_list
        

In [16]:
TOKENS_OUTPUT_POST = 80
TOKENS_OUTPUT_HEADER = 150

def AreThereValuesInThePost(instruction, instruction_end, system_prompt, post_list_inline, model, post_num:int=None):
    list_labels=[]
    user_content = instruction + str(post_list_inline) + instruction_end
    # messages = [{"role": "user", "content": user_content}]
#     messages = [
#     {"role": "system", "content": system_prompt},
#     {"role": "user", "content": user_content}
# ]
    TABLE_SCHEMA = {
    "type": "object",
    "properties": {
        "r": {
            "type": "array",
            "items": {
                "type": "array",
                "minItems": 11,
                "maxItems": 11,
                "items": {"type": "integer"}
            }
        }
    },
    "required": ["r"]
}

    if post_num is None:
       max_tokens = 1000  #1000
    else:
        max_tokens = TOKENS_OUTPUT_HEADER
        max_tokens += int(1.1*(TOKENS_OUTPUT_POST*post_num))

#     response = client.models.generate_content(
#         model=model,
#         config=types.GenerateContentConfig(
#             system_instruction=system_prompt,
#             #max_output_tokens=max_tokens,
#             temperature=0.1),
#         contents=user_content
# )
    response = client.models.generate_content(
        model=model,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.1,
            response_mime_type="application/json",
            response_schema=TABLE_SCHEMA,   
        ),
        contents=user_content)

    # list_labels.append(response.text)
    list_labels.append(response.parsed)
   
    return list_labels

PLAINTEXT_PREFIX="```plaintext\n"
PREFIX="```\n"
POSTFIX="```"
COLUMNS_ROW = "post_id,Self-direction,Stimulation,Hedonism,Achievement,Power,Security,Conformity,Tradition,Benevolence,Universalism"
columns_ref = set(COLUMNS_ROW.split(",")) 

def PostProcText(t: str, post_num: int, remove_plaintext: bool = True, check_columns: bool = True) -> str:
    
    t = t.strip()
    
    if remove_plaintext and t.startswith(PLAINTEXT_PREFIX) and t.endswith(POSTFIX) and (len(t) > len(PLAINTEXT_PREFIX + POSTFIX)):
        t = t[len(PLAINTEXT_PREFIX):-len(POSTFIX)]
    if remove_plaintext and t.startswith(PREFIX) and t.endswith(POSTFIX) and (len(t) > len(PREFIX + POSTFIX)):
        t = t[len(PREFIX):-len(POSTFIX)]
    if check_columns:
        if t.startswith(COLUMNS_ROW+"\n"):
            return t
        # else:
        # fix columns
        rows = t.split("\n")
        # any col name in 1st row
        if sum(1 for col in columns_ref if col in rows[0]):
            # rm 1sr row
            rows.pop(0)
        # and fix with default one
        rows.insert(0,COLUMNS_ROW)
        return "\n".join(rows)
    return t
            

## RUN 

In [ ]:
model = "gemini-2.5-pro" #"gpt-4"  #gpt-5
# client=client
model_choices_n=1 # number of chat_gpt iterations
model_try_n=5  # !!! how many times each post should be lableled with model
chunk_rnd_size=100 #50 - 


df_responses=[]
model_try_df_responses = {}
not_parsed_responses = []
for i in range(1, model_try_n+1):
    errors_num = 0
    df_chunks = [] # list of df-s with parsed results for chunk of posts
    idx_done=[]  # номера строк, которые уже выпадали в рандомайзере
    while (len(idx_done)<test1.shape[0]) and (errors_num < 10):
        chunk_idx=SelectRandomExcept(test1, idx_done, chunk_rnd_size)
        chunk_dict=dict(zip(chunk_idx, test1.iloc[chunk_idx].text.to_list()))
       
        posts_chunk_inline, chunk_idx_list=create_post_list_inline(instruction, 
                                                                  instruction_end,
                                                                   chunk_dict, 
                                                                   model)

        response_check_status = True
        try:
            print(chunk_idx_list)
            print(posts_chunk_inline)
            model_responses=AreThereValuesInThePost(instruction, 
                                                    instruction_end,
                                                    system_prompt_final,
                                                    posts_chunk_inline,
                                                    model,
                                                    len(chunk_idx_list),
                                                   )
            print(model_responses)
        except Exception as e:
            print('Error while promt model:', repr(e))
            print(posts_chunk_inline)
            response_check_status = False
        try:
            if response_check_status:
                parsed = model_responses[0]  #new
                rows = parsed["r"] #new
                df_chunk_response = pd.DataFrame(rows, columns=COLUMNS_ROW.split(","))  #new

               
                df_chunk_response["post_id"] = df_chunk_response["post_id"].astype(int)  #new
                df_chunk_response = df_chunk_response.set_index("post_id")  #new
             
                if df_chunk_response.isna().any().any():
                    raise ValueError("NaN in chunk output")

                if set(df_chunk_response.index) != set(chunk_idx_list):  #new
                    raise ValueError("Returned post_id set does not match requested chunk")  #new

                df_chunk_response = df_chunk_response.reindex(chunk_idx_list)  #new

        
        except Exception as e:
            response_check_status = False
            if model_responses is not None:
                not_parsed_responses.append(model_responses[0])
            print("Error to parse model response as structured output:", repr(e))
        
        if response_check_status:
            df_chunk_response = df_chunk_response.dropna(axis=0)
            response_check_status = (df_chunk_response.empty is False)
        if not response_check_status:
            errors_num += 1
            print("Error to parse model response.")
        else:
            errors_num = 0
            df_chunks.append(df_chunk_response)
            print(df_chunk_response)
            idx_done = idx_done + list(df_chunk_response.index)  # 

    if len(df_chunks):
        model_try_df_responses[i] = pd.concat(df_chunks, axis=0).sort_index()

# create multiIndex df for Values columns
if len(model_try_df_responses):
    df_responses = pd.concat(
        model_try_df_responses.values(),
        keys=model_try_df_responses.keys(),
        names=["try"],
        axis=1
    )
else:
    df_responses = None



In [ ]:
df_concat=pd.concat([test1, pd.concat([df_responses[i] for i in range (1,model_try_n+1)]).astype(str).groupby(level=0).agg(','.join)], axis=1)
df_concat


